# 08 · Agents and sandboxing

**Story so far:** the model is live behind an API. The final piece of new functionality:
an **agent** that triages incoming reviews: deciding what to do, calling the model, and
executing LLM-generated code safely. Agentic workloads run **untrusted code**: an LLM
writes Python, and something has to execute it without trusting it. Union ships two
complementary sandboxes for this, plus the building blocks (tasks, traces, apps)
that turn an agent loop into production infrastructure.

**Learning goals**

1. Run arbitrary (LLM-generated) code in an ephemeral, isolated container with
   `flyte.sandbox.create`
2. Run LLM-generated *orchestration* in a Monty workflow sandbox: the
   **programmatic tool calling / code mode** pattern
3. Assemble Review Radar's triage agent from primitives you already know
   (`@env.task` + `@flyte.trace`)
4. Know where the rest of the agent stack lives (HITL, memory, chat UI, frameworks)

> ⚠️ **Experimental:** `flyte.sandbox` is alpha; APIs may change between SDK releases.
> Pin your SDK version (these notebooks pin 2.5.7) and re-verify on upgrade.

In [ ]:
import flyte
import flyte.sandbox

flyte.init_from_config()

## 1. Why sandbox LLM-generated code

The model has no intent, but it can still emit `DELETE FROM orders WHERE 1=1`, an
infinite loop, or code that reads your env vars and posts them somewhere. Running it
unsandboxed means trusting the model never to make those mistakes. Union's two
approaches:

| | **Code sandbox** | **Workflow sandbox (Monty)** |
|---|---|---|
| Runtime | Ephemeral container, built on demand, discarded after one run | Rust-based sandboxed Python interpreter, in-process |
| Startup | Seconds (image build is cached) | Microseconds |
| Capabilities | Full Python: any package, file I/O, shell | Pure control flow only: no imports, no I/O, no network |
| Use for | Arbitrary computation: data crunching, test execution | LLM-written *orchestration* that calls your registered tools |
| State | Stateless per invocation | Runs alongside the worker process |

## 2. Code sandbox: `flyte.sandbox.create`

**Auto-IO mode** (default): declare typed inputs/outputs, write only the business logic.
Flyte generates the argument parsing, exposes inputs as variables, and captures any
variable whose name matches a declared output. Dependencies are declared per-sandbox and
the image is built through the deployment's Image Builder, then cached by content.

In [ ]:
sandbox = flyte.sandbox.create(
    name="double",
    code="result = x * 2",
    inputs={"x": int},
    outputs={"result": int},
)

result = await sandbox.run.aio(x=21)
result

Now the story case: an analyst asks Review Radar's agent for a custom statistic, the LLM
writes pandas code for it, and it runs where it can't touch anything:

In [ ]:
# Pretend an LLM produced this for: "average stars for reviews mentioning 'broken'"
llm_generated = '''
import json
import pandas as pd
rows = [json.loads(line) for line in corpus_jsonl.splitlines()]
df = pd.DataFrame(rows)
hits = df[df["text"].str.contains("broken", case=False)]
avg_stars = float(hits["stars"].mean()) if len(hits) else 0.0
n_matches = int(len(hits))
'''

stats_sandbox = flyte.sandbox.create(
    name="llm-corpus-stats",
    code=llm_generated,
    inputs={"corpus_jsonl": str},
    outputs={"avg_stars": float, "n_matches": int},
    packages=["pandas"],                # sandbox image deps, built & cached on demand
    timeout=120,                        # hard stop for runaway code
    resources=flyte.Resources(cpu="1", memory="1Gi"),
)

demo_corpus = "\n".join([
    '{"id": 1, "stars": 1, "text": "terrible shoes, arrived broken"}',
    '{"id": 2, "stars": 5, "text": "absolutely love this espresso machine"}',
    '{"id": 3, "stars": 2, "text": "broken after two days"}',
])

avg_stars, n_matches = await stats_sandbox.run.aio(corpus_jsonl=demo_corpus)
print(f"avg_stars={avg_stars} n_matches={n_matches}")

Two more modes when auto-IO doesn't fit: **verbatim** (`auto_io=False`, a complete
script managing its own I/O via `/var/inputs/` and `/var/outputs/`) and **command**
(`command=[...]`, any binary or shell pipeline). Useful knobs: `system_packages`,
`env_vars`, `secrets`, `retries`, `cache="auto"`.

## 3. Workflow sandbox: programmatic tool calling (code mode)

Sequential tool calling routes every intermediate result through the model's context:
token-expensive and slow at each round-trip. **Code mode** flips it: the LLM writes *one
program* that calls your tools; the sandbox executes loops/conditionals/data-shuffling
locally and only the final answer returns to the model.

Union implements this with **Monty**: your tools are ordinary `@env.task` workers (full
containers, full capabilities), and the LLM-generated orchestration runs in an
interpreter that structurally cannot import, do I/O, or touch the network. It can only
call the tools you registered.

In [ ]:
env = flyte.TaskEnvironment(
    name="code_mode",
    resources=flyte.Resources(cpu="1", memory="1Gi"),
)


# Tools: ordinary tasks, run in their own containers with full capabilities
@env.task
def count_negative(stars: list[int]) -> int:
    return sum(1 for s in stars if s <= 2)


@env.task
def escalation_ratio(negatives: int, total: int) -> float:
    return round(negatives / max(total, 1), 3)


# Static form: sandboxed orchestrator you write yourself
@env.sandbox.orchestrator
def triage_stats(stars: list[int]) -> float:
    negatives = count_negative(stars)          # each call runs the worker container
    return escalation_ratio(negatives, len(stars))


run = flyte.run(triage_stats, stars=[1, 5, 2, 4, 1, 3])
print(run.url)
run.wait()
run.outputs()

The dynamic form is the agent-facing one: the orchestration code arrives as a string
(from an LLM) and the value of the last expression becomes the output:

In [ ]:
# Pretend an LLM wrote this plan for: "what share of these reviews need escalation?"
llm_plan = '''
negatives = count_negative(stars)
escalation_ratio(negatives, total)
'''

compute = flyte.sandbox.orchestrator_from_str(
    llm_plan,
    inputs={"stars": list[int], "total": int},
    output=float,
    tasks=[count_negative, escalation_ratio],   # the ONLY capabilities the code has
    name="llm-triage-plan",
)

run = flyte.run(compute, stars=[1, 5, 2, 4, 1, 3], total=6)
print(run.url)
run.wait()
run.outputs()

Notice what you get because the plan executed *on Union*: every tool call is a tracked
action (durable, retryable, cached per your task config), the whole plan is observable in
the UI, and inputs/outputs are typed and validated at the sandbox boundary. That's the
difference between this and an `exec()` in your agent process.

## 4. The Review Radar triage agent

Union is framework-agnostic: an agent loop is just Python inside a task, using any LLM
SDK. The platform's role is the production layer: each agent step is a **sandboxed
container** (`@env.task`), every LLM/tool call is a **traced checkpoint** (`@flyte.trace`,
from [03](./03-processing-at-scale.ipynb) §5), and the loop's state is durable.

The triage agent below uses a deterministic mock "LLM" so it runs without an API key.
Swap `reason()`'s body for a real chat-completions call (inject the key via
`flyte.Secret`, or point the client at chapter 07's vLLM endpoint for a fully self-hosted
stack) and nothing else changes. Its `score_review` tool calls the live model API from
chapter 07.

In [ ]:
import json

agent_env = flyte.TaskEnvironment(
    name="triage_agent",
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    image=flyte.Image.from_debian_base(name="workshop-agent", python_version=(3, 12))
    .with_pip_packages("httpx"),
    # For a real LLM: secrets=[flyte.Secret(key="LLM_API_KEY", as_env_var="LLM_API_KEY")]
)


@flyte.trace
async def reason(review: str, history: list[str]) -> dict:
    """The 'LLM' decides the next action. Mocked as a fixed policy; in production this
    is one chat-completions call, and the decorator gives it a trace + checkpoint."""
    if not any(h.startswith("score:") for h in history):
        return {"action": "score_review", "input": review}
    score = float(history[-1].removeprefix("score:"))
    verdict = "escalate to support" if score < 0.4 else "auto-acknowledge"
    return {"action": "finish", "input": verdict}


@flyte.trace
async def act(action: str, action_input: str, model_api: str) -> str:
    """Tool dispatch. Tools can be plain code, other tasks, sandboxes, or live APIs."""
    if action == "score_review":
        if model_api.startswith("https://"):
            import httpx
            async with httpx.AsyncClient() as client:
                r = await client.post(f"{model_api}/score",
                                      json={"text": action_input}, timeout=60)
                return f"score:{r.json()['positive_probability']}"
        # offline fallback so the chapter runs without 07 deployed
        return f"score:{0.2 if 'broken' in action_input else 0.8}"
    raise ValueError(f"unknown tool {action}")


@agent_env.task
async def triage(review: str, model_api: str = "", max_steps: int = 5) -> str:
    history: list[str] = []
    for _ in range(max_steps):
        decision = await reason(review, history)                    # traced checkpoint
        if decision["action"] == "finish":
            return decision["input"]
        history.append(await act(decision["action"], decision["input"], model_api))
    return json.dumps({"error": "max_steps reached", "history": history})


# Pass APP_URL from chapter 07 as model_api to use the live model:
run = flyte.run(triage, review="terrible trail shoes, arrived broken")
print(run.url)
run.wait()
run.outputs()

Open the run URL: the `reason`/`act` calls appear as traced spans with their
inputs/outputs. If the pod died mid-loop, the retry would resume from the last completed
trace instead of re-running the earlier LLM calls.

## 5. The rest of the agent stack

Everything else an agent needs maps to something this series already covered:

| Agent need | Union feature | Covered in |
|---|---|---|
| Human-in-the-loop approval | `flyte.new_condition` pauses the loop for a typed signal | [03](./03-processing-at-scale.ipynb) §6 |
| Low-latency agent steps | `ReusePolicy` warm pools (clients stay loaded) | [05](./05-reusable-containers.ipynb) |
| Parallel sub-agents / plan-and-execute | `asyncio.gather` fan-out, each sub-task its own container | [03](./03-processing-at-scale.ipynb) §1 |
| Chat UI / agent as a service | Apps (`AppEnvironment`, FastAPI/WebSocket) | [07](./07-serving.ipynb) |
| LLM API keys | `flyte.Secret` | [02](./02-data-flow.ipynb) §3 |
| Self-hosted models for agents | vLLM app (OpenAI-compatible) | [07](./07-serving.ipynb) §3 |

Beyond hand-rolled loops, the docs' **Build an agent** section covers the
batteries-included *Flyte Agent harness* (tool registry, MCP servers, memory, HITL), the
*agent chat UI*, and integrations that run **LangGraph / PydanticAI / OpenAI Agents SDK**
agents on Union with durability added underneath.

> 💬 **Discuss:** which agent use cases are on the customer's roadmap, and where would
> programmatic tool calling cut the most tokens/latency compared with sequential tool
> calling as they run it today?

**Story complete.** Raw reviews → durable datasets → parallel processing → cached
features → warm-pool scoring → a trained model → a live API → an agent on top. Every
chapter used the same handful of primitives; they compose.

## Further reading

- Union docs: [Sandboxing](https://www.union.ai/docs/v2/union/user-guide/sandboxing/) · [Build an agent](https://www.union.ai/docs/v2/union/user-guide/build-agent/)
- For teams with an existing v1 estate: [09-migration-v1-to-v2](./09-migration-v1-to-v2.ipynb)